# 02 Độ nhạy khi giữ đủ đơn

So sánh A (quy tắc v4, loại đơn >24 giờ) và B (giữ đủ đơn, làm sạch GPS và giữ cờ như A). Đây là đánh giá độ nhạy với nhãn nhu cầu được định nghĩa bởi hai chế độ lọc, không phải khẳng định chất lượng thời gian giao của 8 đơn.

## Tiêu chí đặt trước khi chạy

Cấu hình tại `configs/sensitivity_keep_all_orders.json` được ghi trước lần chạy đầu: thứ hạng đầy đủ theo WAPE validation không đổi, và thay đổi MAE tương đối tuyệt đối **nhỏ hơn 1% cho mọi baseline**. Khi MAE của A bằng 0, B cũng phải bằng 0. Cùng cửa sổ A, cùng danh mục chuỗi và cùng ô đủ điều kiện; A và B dùng lịch sử và nhãn riêng theo chế độ. Không dùng test để xác nhận tiêu chí.

Đánh giá riêng cửa sổ chính và toàn kỳ. LightGBM/XGBoost phải lặp lại phép so sánh sau khi được triển khai; kết quả dưới đây chưa bao phủ mô hình học máy hay CVRP.

In [1]:
import sys, json
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(root))
import pandas as pd
from IPython.display import Markdown, display
from src import config
from src.sensitivity_keep_all_orders import main
report = main()
out = config.FORECASTS_DIR / 'sensitivity_keep_all_orders'

               scenario              model    mae_A    mae_B   wape_A   wape_B  relative_mae_change
  primary_stable_period            naive_1 1.847561 1.847561 0.285849 0.285795             0.000000
  primary_stable_period         seasonal_7 1.795122 1.797561 0.277736 0.278061             0.001359
  primary_stable_period     rolling_mean_7 1.371603 1.372997 0.212210 0.212386             0.001016
  primary_stable_period    rolling_mean_14 1.420122 1.421341 0.219717 0.219864             0.000859
  primary_stable_period    historical_mean 1.591127 1.591842 0.246174 0.246239             0.000449
  primary_stable_period historical_weekday 1.634248 1.634959 0.252846 0.252908             0.000435
sensitivity_full_period            naive_1 1.687231 1.687231 0.268034 0.268004             0.000000
sensitivity_full_period         seasonal_7 1.747489 1.748924 0.277607 0.277803             0.000821
sensitivity_full_period     rolling_mean_7 1.304263 1.305083 0.207196 0.207302             0.000629


## 1 Kiểm tra tám đơn và tổng đơn theo chuỗi

In [2]:
display(pd.read_csv(out / 'retained_8_orders.csv'))
display(pd.read_csv(out / 'orders_by_series.csv'))
display(pd.DataFrame({'A': [report['orders_A']], 'B': [report['orders_B']]}, index=['total orders']))

,order_id,region_id,aoi_type,ds_date,delivery_minutes,flag_invalid_gps
0,1249828,129,1,2022-07-28,2948.0,False
1,3733224,129,1,2022-08-28,1468.0,False
2,1279740,129,1,2022-06-18,1537.0,False
3,331475,129,1,2022-07-13,1477.0,False
4,1969323,129,1,2022-06-08,1453.0,False
5,3881643,129,1,2022-07-04,2093.0,False
6,3321484,129,14,2022-07-10,1641.0,False
7,420775,129,14,2022-07-03,3573.0,False


,region_id,aoi_type,A,B,B_minus_A
0,31,0,72,72,0
1,31,1,5853,5853,0
2,31,3,27,27,0
3,31,4,59,59,0
4,31,7,444,444,0
5,31,10,15,15,0
6,31,12,42,42,0
7,31,13,67,67,0
8,31,14,2239,2239,0
9,53,0,167,167,0


,A,B
total orders,31407,31415


## 2 MAE WAPE và thứ hạng validation

Các cột A/B được đặt cạnh nhau; WAPE lưu dưới dạng tỷ lệ, không phải phần trăm.

In [3]:
comparison = pd.read_csv(out / 'validation_comparison.csv')
for scenario, sub in comparison.groupby('scenario'):
    print(scenario)
    table = sub.set_index('model')[['mae_A','mae_B','wape_A','wape_B']].copy()
    table.columns = pd.MultiIndex.from_tuples([('MAE','A'),('MAE','B'),('WAPE','A'),('WAPE','B')])
    display(table)
    display(sub[['model','relative_mae_change','mae_within_threshold']])
display(pd.DataFrame(report['results']))

primary_stable_period


MAE                WAPE          
                           A         B         A         B
model                                                     
naive_1             1.847561  1.847561  0.285849  0.285795
seasonal_7          1.795122  1.797561  0.277736  0.278061
rolling_mean_7      1.371603  1.372997  0.212210  0.212386
rolling_mean_14     1.420122  1.421341  0.219717  0.219864
historical_mean     1.591127  1.591842  0.246174  0.246239
historical_weekday  1.634248  1.634959  0.252846  0.252908

,model,relative_mae_change,mae_within_threshold
0,naive_1,0.000000,True
1,seasonal_7,0.001359,True
2,rolling_mean_7,0.001016,True
3,rolling_mean_14,0.000859,True
4,historical_mean,0.000449,True
5,historical_weekday,0.000435,True


sensitivity_full_period


MAE                WAPE          
                           A         B         A         B
model                                                     
naive_1             1.687231  1.687231  0.268034  0.268004
seasonal_7          1.747489  1.748924  0.277607  0.277803
rolling_mean_7      1.304263  1.305083  0.207196  0.207302
rolling_mean_14     1.379996  1.380713  0.219227  0.219316
historical_mean     1.857948  1.857188  0.295154  0.295000
historical_weekday  1.924722  1.923885  0.305762  0.305594

,model,relative_mae_change,mae_within_threshold
6,naive_1,0.000000,True
7,seasonal_7,0.000821,True
8,rolling_mean_7,0.000629,True
9,rolling_mean_14,0.000520,True
10,historical_mean,0.000409,True
11,historical_weekday,0.000434,True


,scenario,ranking_A,ranking_B,ranking_unchanged,all_mae_within_threshold,passed
0,primary_stable_period,"[rolling_mean_7, rolling_mean_14, historical_m...","[rolling_mean_7, rolling_mean_14, historical_m...",True,True,True
1,sensitivity_full_period,"[rolling_mean_7, rolling_mean_14, naive_1, sea...","[rolling_mean_7, rolling_mean_14, naive_1, sea...",True,True,True


## 3 Kết luận tự động theo tiêu chí

In [4]:
passed = all(item['passed'] for item in report['results'])
maxChange = comparison.relative_mae_change.max() * 100
text = (f'Thứ hạng baseline không đổi và mức thay đổi MAE lớn nhất là {maxChange:.3f}%, dưới ngưỡng 1%. Quy tắc loại 8 đơn không làm thay đổi kết luận lựa chọn baseline trên validation ở hai cửa sổ đã xét.' if passed else 'Có ít nhất một tiêu chí không đạt. Chưa thể kết luận quy tắc loại 8 đơn không ảnh hưởng kết luận baseline; xem bảng chi tiết.')
display(Markdown(text + ' Kết luận giới hạn ở baseline và giao thức này; cần chạy lại với LightGBM/XGBoost sau khi triển khai.'))

Thứ hạng baseline không đổi và mức thay đổi MAE lớn nhất là 0.136%, dưới ngưỡng 1%. Quy tắc loại 8 đơn không làm thay đổi kết luận lựa chọn baseline trên validation ở hai cửa sổ đã xét. Kết luận giới hạn ở baseline và giao thức này; cần chạy lại với LightGBM/XGBoost sau khi triển khai.